# Merge: Clinical + RNA

Merges the same clinical dataset with RNA MB-selected features for each of 8 RNA datasets.

**Notebook location:** `01_Causal_feature_extraction/MERGE/`  
**Output:** `MERGE/01_ultra_conservative.csv` ... `MERGE/08_composite.csv`

## Imports & Paths

In [3]:
"""
MERGE: Clinical + RNA (best MB configuration per dataset)

For each of 8 RNA datasets:
  1. Read summary_all_results.csv -> pick best C-index (algorithm + alpha)
  2. Load gene list from mb_results/{dataset}/{algo}_alpha{alpha}_genes.txt
  3. Load statistical_filtered CSV, keep only selected genes
  4. Normalize RNA IDs to patient level (TCGA-XX-XXXX), filter to Primary Tumor
  5. Merge with clinical (CLIN_ prefix already set) + RNA (RNA_ prefix added here)
  6. Save to MERGE/ with short names: 01_ultra_conservative.csv, etc.

Script location: 01_Causal_feature_extraction/
Output:          01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS  — relative to this script (no hardcoded user paths)
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent        # 01_Causal_feature_extraction/
except NameError:
    # Jupyter: if notebook is inside MERGE/, go one level up
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

RNA_DIR   = SCRIPT_DIR / "RNA"
FILT_DIR  = RNA_DIR / "statistical_filtered"
MB_DIR    = RNA_DIR / "mb_results"
CLIN_FILE = SCRIPT_DIR / "Clinical" / "clinical_preprocessed_prefixed.csv"
OUT_DIR   = SCRIPT_DIR / "MERGE"
OUT_DIR.mkdir(exist_ok=True)

print(f"Script dir:    {SCRIPT_DIR}")
print(f"Clinical:      {CLIN_FILE.exists()} -> {CLIN_FILE.name}")
print(f"MB results:    {MB_DIR.exists()}")
print(f"Output:        {OUT_DIR}")

# ---------------------------------------------------------------------------
# SHORT NAMES for output files  (same order as RNA datasets 1-8)
# ---------------------------------------------------------------------------
DATASET_SHORT_NAMES = {
    "rna_1_ultra_conservative_723genes": "01_ultra_conservative",
    "rna_2_conservative_1079genes":      "02_conservative",
    "rna_3_standard_1751genes":          "03_standard",
    "rna_4_fdr_significant_1000genes":   "04_fdr_significant",
    "rna_5_balanced_1066genes":          "05_balanced",
    "rna_6_correlation_1081genes":       "06_correlation",
    "rna_7_top_correlated_200genes":     "07_top_correlated",
    "rna_8_composite_1000genes":         "08_composite",
}


# ---------------------------------------------------------------------------

Script dir:    C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
Clinical:      True -> clinical_preprocessed_prefixed.csv
MB results:    True
Output:        C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE


## Step 1 — Load Clinical  (shared across all datasets)

In [5]:
# STEP 1: LOAD CLINICAL  (same for all 8 datasets)
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 1: LOAD CLINICAL  (shared across all 8 datasets)")
print("="*70)

clinical = pd.read_csv(CLIN_FILE, index_col=0)
print(f"Shape:   {clinical.shape}")
print(f"Index:   {clinical.index[:3].tolist()}")
print(f"Prefix:  {clinical.columns[0][:5]}...  (should be CLIN_)")

# ---------------------------------------------------------------------------


STEP 1: LOAD CLINICAL  (shared across all 8 datasets)
Shape:   (1098, 144)
Index:   ['TCGA-BH-A0W3', 'TCGA-AR-A24V', 'TCGA-E9-A1NE']
Prefix:  CLIN_...  (should be CLIN_)


## Step 2 — Select Best RNA Config per Dataset

In [7]:
# STEP 2: SELECT BEST RNA CONFIG PER DATASET
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 2: SELECT BEST RNA CONFIG PER DATASET (highest C-index)")
print("="*70)

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")

best = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index", "n_causal_genes"]]

print(best.to_string(index=False))

# ---------------------------------------------------------------------------


STEP 2: SELECT BEST RNA CONFIG PER DATASET (highest C-index)
                          dataset algorithm  alpha  c_index  n_causal_genes
    rna_7_top_correlated_200genes      GSMB   0.05 0.611755              50
  rna_4_fdr_significant_1000genes      GSMB   0.05 0.558697              50
        rna_8_composite_1000genes      IAMB   0.05 0.553179              50
      rna_6_correlation_1081genes      GSMB   0.05 0.537231              50
         rna_5_balanced_1066genes      GSMB   0.05 0.511930              50
         rna_3_standard_1751genes      IAMB   0.10 0.459153              50
rna_1_ultra_conservative_723genes      IAMB   0.10 0.456804              50
     rna_2_conservative_1079genes      IAMB   0.20 0.444775              50


## Step 3 — Helper Functions

In [9]:
# HELPERS
# ---------------------------------------------------------------------------
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = MB_DIR / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found for {dataset} / {algorithm} / alpha={alpha}"
    )


def merge_one(clinical_df, dataset, algorithm, alpha):
    # find statistical_filtered file
    candidates = list(FILT_DIR.glob(f"{dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No statistical_filtered file for '{dataset}'")
    rna_file = candidates[0]

    genes     = load_genes(dataset, algorithm, alpha)
    rna       = pd.read_csv(rna_file, index_col=0)
    available = [g for g in genes if g in rna.columns]
    missing_g = [g for g in genes if g not in rna.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} genes missing from RNA file")
    rna = rna[available].copy()
    print(f"  RNA loaded: {rna.shape}  ({rna_file.name})")

    # Primary Tumor only (-01A, -01B, etc.)
    tumor_mask = rna.index.str.contains(r"-01[A-Z]?$", regex=True)
    rna = rna[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(rna)} samples")

    # normalize IDs to patient level
    rna.index = [normalize_id(i) for i in rna.index]

    # average duplicates
    if rna.index.duplicated().any():
        n_pts = rna.index[rna.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate samples")
        rna = rna.groupby(rna.index).mean()

    # add RNA_ prefix
    rna.columns = [f"RNA_{c}" for c in rna.columns]

    common = sorted(set(clinical_df.index) & set(rna.index))
    print(f"  Common patients: {len(common)}  "
          f"(clinical={len(clinical_df)}, RNA={len(rna)})")
    if not common:
        raise ValueError("No common patients — check ID format")

    merged = pd.concat([clinical_df.loc[common], rna.loc[common]], axis=1)

    # verification
    n_clin = clinical_df.shape[1]
    n_rna  = rna.shape[1]
    assert merged.shape == (len(common), n_clin + n_rna), "Shape mismatch!"
    assert merged.isna().sum().sum() == 0, \
        f"Missing values: {merged.isna().sum().sum()}"
    clin_cols = [c for c in merged.columns if c.startswith("CLIN_")]
    rna_cols  = [c for c in merged.columns if c.startswith("RNA_")]
    print(f"  CLIN_ cols: {len(clin_cols)}  |  RNA_ cols: {len(rna_cols)}  |  "
          f"total: {merged.shape[1]}  |  no missing values")

    return merged

# ---------------------------------------------------------------------------

## Step 4 — Merge All 8 Datasets

In [11]:
# STEP 3: MERGE ALL 8 DATASETS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: MERGE ALL 8 DATASETS")
print("="*70)

results = []

for _, row in best.iterrows():
    dataset   = row["dataset"]
    algorithm = row["algorithm"]
    alpha     = float(row["alpha"])
    c_index   = float(row["c_index"])

    short_name = DATASET_SHORT_NAMES.get(dataset, dataset)

    print(f"\n{'─'*70}")
    print(f"[{short_name}]  {dataset}")
    print(f"Config: {algorithm}  alpha={alpha}  C-index={c_index:.4f}")

    try:
        merged   = merge_one(clinical, dataset, algorithm, alpha)
        out_file = OUT_DIR / f"{short_name}.csv"
        merged.to_csv(out_file)
        print(f"  Saved: {out_file.name}  {merged.shape}")

        results.append({
            "file":                short_name + ".csv",
            "dataset":             dataset,
            "algorithm":           algorithm,
            "alpha":               alpha,
            "c_index":             c_index,
            "n_samples":           merged.shape[0],
            "n_clinical_features": sum(1 for c in merged.columns if c.startswith("CLIN_")),
            "n_rna_features":      sum(1 for c in merged.columns if c.startswith("RNA_")),
            "n_total_features":    merged.shape[1],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()

# ---------------------------------------------------------------------------


STEP 3: MERGE ALL 8 DATASETS

──────────────────────────────────────────────────────────────────────
[07_top_correlated]  rna_7_top_correlated_200genes
Config: GSMB  alpha=0.05  C-index=0.6118
  RNA loaded: (1073, 50)  (rna_7_top_correlated_200genes.csv)
  After Primary Tumor filter: 1071 samples
  Common patients: 1071  (clinical=1098, RNA=1071)
  CLIN_ cols: 144  |  RNA_ cols: 50  |  total: 194  |  no missing values
  Saved: 07_top_correlated.csv  (1071, 194)

──────────────────────────────────────────────────────────────────────
[04_fdr_significant]  rna_4_fdr_significant_1000genes
Config: GSMB  alpha=0.05  C-index=0.5587
  RNA loaded: (1073, 50)  (rna_4_fdr_significant_1000genes.csv)
  After Primary Tumor filter: 1071 samples
  Common patients: 1071  (clinical=1098, RNA=1071)
  CLIN_ cols: 144  |  RNA_ cols: 50  |  total: 194  |  no missing values
  Saved: 04_fdr_significant.csv  (1071, 194)

──────────────────────────────────────────────────────────────────────
[08_composite]  rn

## Step 5 — Summary & Verification

In [13]:
# STEP 4: SUMMARY + VERIFICATION
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(OUT_DIR / "merge_summary.csv", index=False)

    print(f"\nAll 8 datasets share the same clinical: "
          f"{df['n_clinical_features'].nunique() == 1} "
          f"({df['n_clinical_features'].iloc[0]} CLIN_ features)")
    print(f"RNA features differ across datasets:    "
          f"{df['n_rna_features'].nunique() > 1}  "
          f"(range {df['n_rna_features'].min()}–{df['n_rna_features'].max()})")
    print(f"Samples range: {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nSaved to: {OUT_DIR}")


STEP 4: SUMMARY
                     file                           dataset algorithm  alpha  c_index  n_samples  n_clinical_features  n_rna_features  n_total_features
    07_top_correlated.csv     rna_7_top_correlated_200genes      GSMB   0.05 0.611755       1071                  144              50               194
   04_fdr_significant.csv   rna_4_fdr_significant_1000genes      GSMB   0.05 0.558697       1071                  144              50               194
         08_composite.csv         rna_8_composite_1000genes      IAMB   0.05 0.553179       1071                  144              50               194
       06_correlation.csv       rna_6_correlation_1081genes      GSMB   0.05 0.537231       1071                  144              50               194
          05_balanced.csv          rna_5_balanced_1066genes      GSMB   0.05 0.511930       1071                  144              50               194
          03_standard.csv          rna_3_standard_1751genes      IAMB  